In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
from math import sqrt

df = pd.read_csv("movies.csv", sep='\t', names=['user_id','movie_id','rating','timestamp'])

user_item_matrix = df.pivot(index='user_id', columns='movie_id', values='rating')
user_item_matrix_filled = user_item_matrix.fillna(0)
user_ids = user_item_matrix.index.tolist()

user_similarity = cosine_similarity(user_item_matrix_filled)
user_similarity_df = pd.DataFrame(user_similarity, index=user_ids, columns=user_ids)

def get_top_n_similar_users(user_id, n=5):
    top_users = user_similarity_df.loc[user_id].sort_values(ascending=False).drop(user_id)
    return top_users.head(n).index

def predict_ratings(user_id):
    top_users = get_top_n_similar_users(user_id, n=5)
    sim_scores = user_similarity_df.loc[user_id, top_users]
    weighted_ratings = user_item_matrix_filled.loc[top_users].T.dot(sim_scores)
    sum_sims = sim_scores.sum()
    predicted_ratings = weighted_ratings / sum_sims if sum_sims != 0 else weighted_ratings
    predicted_ratings = predicted_ratings.reindex(user_item_matrix.columns)
    return predicted_ratings

user_id_example = user_ids[0]
pred_ratings = predict_ratings(user_id_example)
top_n_recommendations = pred_ratings.sort_values(ascending=False).head(10)
top_n_recommendations = top_n_recommendations[top_n_recommendations > 0]

actual_ratings = user_item_matrix.loc[user_id_example].fillna(0)
rmse = sqrt(mean_squared_error(actual_ratings, pred_ratings))
mae = mean_absolute_error(actual_ratings, pred_ratings)

plt.figure(figsize=(12,8))
sns.heatmap(user_item_matrix_filled, cmap="YlGnBu")
plt.title("User-Item Matrix")
plt.show()

plt.figure(figsize=(12,8))
sns.heatmap(user_similarity_df, cmap="coolwarm")
plt.title("User Similarity Matrix")
plt.show()

if not top_n_recommendations.empty:
    top_n_recommendations.plot(kind='bar', title="Top-N Recommended Movies")
    plt.show()
else:
    print("No recommendations available for this user.")

print("RMSE:", rmse)
print("MAE:", mae)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from math import sqrt

df = pd.read_csv("movies.csv", sep='\t', names=['user_id','movie_id','rating','timestamp'])

item_user_matrix = df.pivot(index='movie_id', columns='user_id', values='rating')
item_user_matrix_filled = item_user_matrix.fillna(0)
user_ids = item_user_matrix.columns.tolist()
movie_ids = item_user_matrix.index.tolist()

item_similarity = cosine_similarity(item_user_matrix_filled)
item_similarity_df = pd.DataFrame(item_similarity, index=movie_ids, columns=movie_ids)

def get_top_n_similar_items(item_id, n=5):
    top_items = item_similarity_df.loc[item_id].sort_values(ascending=False).drop(item_id)
    return top_items.head(n).index

def recommend_items_for_user(user_id, top_n=10):
    rated_movies = item_user_matrix[item_user_matrix[user_id] > 0].index.tolist()
    recommendations = pd.Series(dtype=float)
    for movie in rated_movies:
        similar_items = get_top_n_similar_items(movie, n=5)
        for sim_movie in similar_items:
            if sim_movie not in rated_movies:
                recommendations[sim_movie] = recommendations.get(sim_movie, 0) + item_similarity_df.loc[movie, sim_movie]
    recommendations = recommendations.sort_values(ascending=False).head(top_n)
    return recommendations[recommendations > 0]

user_id_example = user_ids[0]
item_recommendations = recommend_items_for_user(user_id_example)

actual_ratings = item_user_matrix[user_id_example].fillna(0)
rmse_item = sqrt(mean_squared_error(actual_ratings, item_user_matrix_filled[user_id_example]))

precision_at_k = len(set(item_recommendations.index) & set(df[df['user_id']==user_id_example]['movie_id'])) / 10

plt.figure(figsize=(12,8))
sns.heatmap(item_user_matrix_filled, cmap="YlGnBu")
plt.title("Item-User Matrix")
plt.show()

plt.figure(figsize=(12,8))
sns.heatmap(item_similarity_df, cmap="coolwarm")
plt.title("Item Similarity Matrix")
plt.show()

if not item_recommendations.empty:
    item_recommendations.plot(kind='bar', title="Top Similar Items Recommendations")
    plt.show()
else:
    print("No recommendations available for this user.")

print("RMSE:", rmse_item)
print("Precision@10:", precision_at_k)